# 3D Census Maps: Chicago Commuting Patterns

This notebook visualizes **Bicycle Commuting** vs. **Remote Work** in Cook County, Illinois (Chicago area) using 2021 ACS Data.

- **Height**: Percentage of workers Commuting by Bicycle
- **Color**: Percentage of workers Working from Home (Blue=Low, Red=High)

In [ ]:
import pandas as pd
import geopandas as gpd
from census import Census
import pydeck as pdk
import numpy as np

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

## 1. Fetch Data (ACS 2021 5-Year)

In [ ]:
# REPLACE WITH YOUR API KEY
CENSUS_API_KEY = "942e0a44c121ca03ced84b727df9b004f1f1367d"
c = Census(CENSUS_API_KEY)

# Variables from Table B08301 (Means of Transportation to Work)
# B08301_001E: Total Workers
# B08301_018E: Bicycle
# B08301_021E: Worked from home
VARIABLES = {
    "B08301_001E": "Total_Workers",
    "B08301_018E": "Bicycle",
    "B08301_021E": "Work_From_Home"
}

try:
    # Fetch data for Cook County, IL (State 17, County 031) by Tract
    print("Fetching ACS 2021 Data for Cook County...")
    census_data = c.acs5.get(("NAME", *VARIABLES.keys()), 
                             {'for': 'tract:*', 'in': 'state:17 county:031', 'year': 2021})
    
    df = pd.DataFrame(census_data)
    
    # Rename columns
    df = df.rename(columns=VARIABLES)
    
    # Convert to numeric
    cols = list(VARIABLES.values())
    for col in cols:
        df[col] = pd.to_numeric(df[col])
        
    # Calculate Percentages
    # Avoid division by zero
    df['Pct_Bike'] = df['Bicycle'] / df['Total_Workers']
    df['Pct_WFH'] = df['Work_From_Home'] / df['Total_Workers']
    
    df = df.fillna(0)
    
    print(f"Fetched {len(df)} tracts.")
    print(df[['NAME', 'Pct_Bike', 'Pct_WFH']].head())

except Exception as e:
    print(f"Error fetching data: {e}")

## 2. Load Geometry & Merge

In [ ]:
# Load Illinois Tracts Geometry
url = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_17_tract_500k.zip"

try:
    print("Loading Geometry...")
    gdf_tracts = gpd.read_file(url)
    
    # Filter for Cook County (031)
    gdf_tracts = gdf_tracts[gdf_tracts['COUNTYFP'] == '031'].copy()
    
    # Merge with Census Data
    # Merge on TRACT CE (GEOID is constructed differently sometimes, best to match components or full GEOID if consistent)
    # API returns 'tract', 'state', 'county'. Shapefile has 'TRACTCE', 'STATEFP', 'COUNTYFP'.
    # Let's use the API 'tract' and shapefile 'TRACTCE'
    
    gdf_tracts['tract_clean'] = gdf_tracts['TRACTCE']
    
    # API dataframe 'tract' column matches TRACTCE
    merged = gdf_tracts.merge(df, left_on='tract_clean', right_on='tract', how='left')
    
    # Reproject to 4326 for PyDeck
    merged = merged.to_crs(epsg=4326)
    
    # Clean NaN
    merged = merged.fillna(0)
    
    print(f"Merged data. Resulting shape: {merged.shape}")

except Exception as e:
    print(f"Error processing geometry: {e}")

## 3. PyDeck Visualization

In [ ]:
# Normalize WFH for Color (Blue to Red)
# 0% -> Blue, 100% (or Max) -> Red
max_wfh = merged['Pct_WFH'].max()
if max_wfh == 0: max_wfh = 1.0

def get_color(wfh_val):
    norm = wfh_val / max_wfh
    norm = max(0.0, min(1.0, norm))
    # Blue (low WFH) -> Red (high WFH)
    # R: 255 * norm
    # B: 255 * (1-norm)
    return [int(255 * norm), 0, int(255 * (1-norm)), 200]

merged['color'] = merged['Pct_WFH'].apply(get_color)

# Define Layer
# Height based on Pct_Bike. Exaggerate height since percentages are small (0.01 etc)
# Scale factor: 50000 -> 1% = 500 meters

layer = pdk.Layer(
    "GeoJsonLayer",
    merged,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=False,
    get_fill_color="color",
    get_line_color=[255, 255, 255, 80],
    get_elevation="Pct_Bike",
    elevation_scale=100000,  # 1% bike commute -> 1000m height (visible)
    auto_highlight=True,
)

# View State (Chicago)
view_state = pdk.ViewState(
    latitude=41.8781,
    longitude=-87.6298,
    zoom=11,
    pitch=45,
    bearing=0
)

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={
        "html": "<b>Tract:</b> {NAME}<br>" 
                "<b>Bike Commuters:</b> {Pct_Bike:.2%}<br>" 
                "<b>Work From Home:</b> {Pct_WFH:.2%}"
    },
    map_style="dark"
)

deck.to_html("chicago_census_3d.html")
deck.show()